# 01. Cancer Evolution, HRD Genomic Scars, Mutational Signatures & ctDNA Liquid Biopsy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gillanilab/pythonic_biotools/blob/main/notebooks/01_cancer_evolution_hrd_and_ctdna.ipynb)

This interactive tutorial walks through an end-to-end somatic cancer genomics workflow in pure Python—without `rpy2` or external R containers—combining six packages from `pythonic_biotools`:

1. **`scarhrd`** — Homologous Recombination Deficiency (`HRD-LOH`, `TAI`, `LST`, and clinical `HRD-sum`) across `settings="default"` vs. `settings="R_default"`.
2. **`mutationtimer`** — Evolutionary timing of somatic point mutations (`clonal [early]`, `clonal [late]`, `subclonal`) and Whole-Genome Doubling (`WGD`).
3. **`tracksig`** — Evolutionary CCF pseudotime ordering and `PELT` changepoint detection across COSMIC mutational signature trajectories.
4. **`mmsig`** — Cosine-pruned mutational signature refitting with multinomial bootstrap confidence intervals.
5. **`ichorcna`** — Ultra-low-pass WGS (`ULP-WGS`) cell-free ctDNA tumor fraction (`TFx`) and copy-number HMM inference across baseline and MRD timepoints.
6. **`conipher`** — Multi-region tumor biopsy `PhyloCCF` clustering and phylogenetic tree enumeration.

In [1]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/gillanilab/pythonic_biotools.git"]
    )
else:
    repo_root = Path.cwd().resolve()
    if not (repo_root / "pythonic_biotools").is_dir() and (repo_root.parent / "pythonic_biotools").is_dir():
        repo_root = repo_root.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from pythonic_biotools import conipher, ichorcna, mmsig, mutationtimer, scarhrd, tracksig

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 15)
print("Loaded oncology toolkit: scarhrd, mutationtimer, tracksig, mmsig, ichorcna, conipher")

Loaded oncology toolkit: scarhrd, mutationtimer, tracksig, mmsig, ichorcna, conipher


## 1. Genomic Scar HRD Scoring (`scarhrd`: `settings="default"` vs. `settings="R_default"`)

We construct allele-specific copy-number segments (`GRCh38`) representing a high-grade serous tumor with Whole-Genome Doubling, copy-neutral LOH, telomeric allelic imbalance, and large-scale state transitions, and compare Pythonic `settings="default"` against strict upstream `sztup/scarHRD` `settings="R_default"`.

In [2]:
cn_segments = pd.DataFrame(
    [
        {"chrom": "chr1", "start": 500_000, "end": 28_000_000, "major_cn": 2, "minor_cn": 0, "total_cn": 2},
        {"chrom": "chr1", "start": 28_000_000, "end": 48_000_000, "major_cn": 3, "minor_cn": 1, "total_cn": 4},
        {"chrom": "chr1", "start": 48_000_000, "end": 120_000_000, "major_cn": 2, "minor_cn": 2, "total_cn": 4},
        {"chrom": "chr2", "start": 400_000, "end": 25_000_000, "major_cn": 2, "minor_cn": 0, "total_cn": 2},
        {"chrom": "chr2", "start": 25_000_000, "end": 90_000_000, "major_cn": 2, "minor_cn": 2, "total_cn": 4},
        {"chrom": "chr17", "start": 1_000_000, "end": 20_000_000, "major_cn": 2, "minor_cn": 0, "total_cn": 2},
        {"chrom": "chr17", "start": 26_000_000, "end": 80_000_000, "major_cn": 3, "minor_cn": 0, "total_cn": 3},
        {"chrom": "chrX", "start": 1_000_000, "end": 32_000_000, "major_cn": 2, "minor_cn": 0, "total_cn": 2},
    ]
)

hrd_default = scarhrd.calculate_hrd(cn_segments, genome="GRCh38", ploidy=3.4, settings="default")
hrd_r = scarhrd.calculate_hrd(cn_segments, genome="GRCh38", ploidy=3.4, settings="R_default")

hrd_comp = pd.DataFrame(
    [
        {
            "settings": "default",
            "HRD_LOH": hrd_default.hrd_loh,
            "TAI": hrd_default.tai,
            "LST": hrd_default.lst,
            "HRD_sum": hrd_default.hrd_sum,
            "LST_ploidy_adj": round(hrd_default.lst_ploidy_adjusted, 2),
        },
        {
            "settings": "R_default",
            "HRD_LOH": hrd_r.hrd_loh,
            "TAI": hrd_r.tai,
            "LST": hrd_r.lst,
            "HRD_sum": hrd_r.hrd_sum,
            "LST_ploidy_adj": round(hrd_r.lst_ploidy_adjusted, 2),
        },
    ]
)
print(hrd_comp.to_string(index=False))

 settings  HRD_LOH  TAI  LST  HRD_sum  LST_ploidy_adj
  default        3    3    3        9             0.0
R_default        2    2    3        7             0.0


## 2. Evolutionary Timing of Somatic SNVs & Whole-Genome Doubling (`mutationtimer`)

Early pre-WGD mutations acquired on a single chromosome copy are duplicated onto $m=2$ copies (`VAF ~ 0.44` at purity `0.80` in `2:2` tetraploid regions), whereas post-WGD HRD mutations occur on $m=1$ copy (`VAF ~ 0.22`). We simulate clock-like `SBS1` early mutations followed by HRD-driven `SBS3` late mutations.

In [3]:
rng = np.random.default_rng(42)
sig_ref = mmsig.make_synthetic_cosmic_reference(["SBS1", "SBS3", "SBS5", "SBS13"])

n_muts = 240
is_early = np.arange(n_muts) < 135
variants_df = pd.DataFrame(
    {
        "chrom": np.where(np.arange(n_muts) < 120, "chr1", "chr2"),
        "pos": rng.integers(50_000_000, 88_000_000, size=n_muts),
        "alt_count": rng.binomial(110, np.where(is_early, 0.44, 0.21)),
        "total_depth": np.full(n_muts, 110),
        "context": np.where(
            is_early,
            rng.choice(tracksig.COSMIC_96_CONTEXTS, size=n_muts, p=sig_ref["SBS1"].to_numpy()),
            rng.choice(tracksig.COSMIC_96_CONTEXTS, size=n_muts, p=sig_ref["SBS3"].to_numpy()),
        ),
    }
)

mt_res = mutationtimer.mutation_time(variants_df, cn_segments, purity=0.80, n_boot=40, seed=42, settings="default")
wgd_res = mutationtimer.estimate_wgd_timing(mt_res.segments_df, settings="default")
wgd_r = mutationtimer.estimate_wgd_timing(
    mutationtimer.mutation_time(variants_df, cn_segments, purity=0.80, n_boot=40, seed=42, settings="R_default").segments_df,
    settings="R_default",
)

print("Timing class distribution:")
print(mt_res.variants_df["timing_class"].value_counts().to_string())
print(
    f"\nWGD detected: {wgd_res.is_wgd} | T_WGD (default) = {wgd_res.wgd_time:.3f} "
    f"[{wgd_res.wgd_time_lo:.3f}, {wgd_res.wgd_time_hi:.3f}] | T_WGD (R_default) = {wgd_r.wgd_time:.3f}"
)

Timing class distribution:
timing_class
clonal [early]    135
clonal [late]     105

WGD detected: True | T_WGD (default) = 0.608 [0.000, 1.000] | T_WGD (R_default) = 0.608


## 3. CCF Pseudotime Signature Trajectories (`tracksig`) & Bootstrap Refitting (`mmsig`)

Next, `tracksig` orders somatic mutations along decreasing Cancer Cell Fraction (`CCF`) pseudotime and detects signature exposure changepoints via `PELT`, while `mmsig` performs cosine-pruned signature refitting with bootstrap confidence intervals.

In [4]:
ts_res = tracksig.tracksig(variants_df, sig_ref, purity=0.80, bin_size=60, settings="default")
print(f"Active signatures: {ts_res.active_signatures} | PELT changepoints (bin indices): {ts_res.changepoints}")
print("\nPer-bin mutational signature trajectory:")
print(ts_res.bin_exposures_df.round(3).to_string())

catalog_df = variants_df["context"].value_counts().reindex(sig_ref.index, fill_value=0).to_frame("TUMOR_01")
mm_def = mmsig.mm_fit_signatures(catalog_df, sig_ref, n_boot=80, seed=42, settings="default")
mm_r = mmsig.mm_fit_signatures(catalog_df, sig_ref, n_boot=80, seed=42, settings="R_default")

print("\nmmsig exposures (settings='default'):")
print(mm_def.exposures.round(3).to_string())
print("\nmmsig exposures (settings='R_default'):")
print(mm_r.exposures.round(3).to_string())

Active signatures: ['SBS1', 'SBS3'] | PELT changepoints (bin indices): [2, 3]

Per-bin mutational signature trajectory:
   bin_id  segment_id  n_mutations  mean_ccf  min_ccf  max_ccf   SBS1   SBS3
0       1           1           60     1.220    1.114    1.477  0.999  0.001
1       2           1           60     1.038    0.955    1.114  0.999  0.001
2       3           2           60     0.689    0.545    0.955  0.181  0.819
3       4           3           60     0.459    0.295    0.523  0.075  0.925

mmsig exposures (settings='default'):
signature   SBS1   SBS3  SBS5  SBS13
sample                              
TUMOR_01   0.571  0.429   0.0    0.0

mmsig exposures (settings='R_default'):
signature   SBS1   SBS3  SBS5  SBS13
sample                              
TUMOR_01   0.567  0.433   0.0    0.0


## 4. Longitudinal Plasma cfDNA Tumor Fraction Estimation (`ichorcna`)

We simulate ultra-low-pass WGS (`ULP-WGS`) 1-Mb plasma cfDNA `logR` profiles at pre-treatment baseline (`true TF = 24%`) and on-therapy minimal residual disease response (`true TF = 4.5%`, `low_tf_mode=True`).

In [5]:
states_cn = np.concatenate([np.full(50, 2), np.full(40, 3), np.full(30, 1)])
chroms_ulp = ["chr1"] * 60 + ["chr2"] * 60
starts_ulp = np.tile(np.arange(0, 60_000_000, 1_000_000), 2)
ends_ulp = starts_ulp + 1_000_000

cfdna_rows = []
for label, tf_true, low_mode in [("Baseline_Plasma", 0.24, False), ("Cycle3_MRD_Plasma", 0.045, True)]:
    exp_logr = np.log2((2.0 * (1.0 - tf_true) + tf_true * states_cn) / 2.0)
    bins_df = pd.DataFrame(
        {
            "chrom": chroms_ulp,
            "start": starts_ulp,
            "end": ends_ulp,
            "logR": exp_logr + rng.normal(0.0, 0.025, size=len(states_cn)),
        }
    )
    fit_ichor = ichorcna.run_ichorcna(bins_df, low_tf_mode=low_mode, max_iter=20, settings="default")
    cfdna_rows.append(
        {
            "timepoint": label,
            "true_TF": tf_true,
            "estimated_TF": round(fit_ichor.tumor_fraction, 4),
            "estimated_ploidy": round(fit_ichor.ploidy, 2),
            "n_segments": len(fit_ichor.segments_df),
        }
    )

print(pd.DataFrame(cfdna_rows).to_string(index=False))

        timepoint  true_TF  estimated_TF  estimated_ploidy  n_segments
  Baseline_Plasma    0.240        0.2418              2.00           4
Cycle3_MRD_Plasma    0.045        0.0297              2.35           4


## 5. Multi-Region Somatic Phylogeny Reconstruction (`conipher`)

Finally, we reconstruct clonal ancestry across three spatially distinct tumor regions (`Primary`, `Met_Liver`, `Met_Lung`) harboring a truncal clone and two region-restricted metastatic branches.

In [6]:
phylo_rows = []
clone_specs = [
    (1.00, 1.00, 1.00, 18),  # Truncal clone
    (0.65, 0.70, 0.00, 12),  # Liver-metastasis shared subclone
    (0.00, 0.00, 0.60, 12),  # Lung-private subclone
]
mut_id = 0
for ccf_p, ccf_li, ccf_lu, count in clone_specs:
    for _ in range(count):
        mut_id += 1
        for region, ccf in (("Primary", ccf_p), ("Met_Liver", ccf_li), ("Met_Lung", ccf_lu)):
            vaf_p = min(0.49, 0.80 * ccf / 2.0)
            phylo_rows.append(
                {
                    "mutation_id": f"MUT_{mut_id:03d}",
                    "region": region,
                    "alt_count": int(rng.binomial(140, vaf_p)),
                    "depth": 140,
                    "major_cn": 1,
                    "minor_cn": 1,
                }
            )

con_res = conipher.run_conipher(
    pd.DataFrame(phylo_rows),
    purity_by_region={"Primary": 0.80, "Met_Liver": 0.80, "Met_Lung": 0.80},
    max_clusters=6,
    min_cluster_size=5,
    seed=42,
    settings="default",
)
print(f"Truncal cluster: {con_res.truncal_cluster} | Enumerated valid trees: {con_res.n_alternative_trees}")
print("\nCluster PhyloCCF per region:")
print(con_res.cluster_ccf_df.round(3).to_string())
print("\nConsensus parent -> child edges:")
print(con_res.parent_child_df.to_string(index=False))

Truncal cluster: 1 | Enumerated valid trees: 1

Cluster PhyloCCF per region:
   cluster_id  n_mutations  is_truncal  Primary  Met_Liver  Met_Lung
0           1           18        True    1.000      1.000     1.000
1           2           12       False    0.632      0.744     0.000
2           3           12       False    0.000      0.000     0.613

Consensus parent -> child edges:
 parent  child
      1      2
      1      3
